# # DL - 지도학습 - Regression 실습

[사용 제한 안내]
> 본 실습 코드는 교육 및 학습 목적으로만 제공됩니다.

[사전 준비 사항]
- 없음

[분석 요건]
- 분석 주제 : 인공신경망을 이용한 주택가격 예측 모델 생성
- 분석 단위 : 주거 지역 (동네 수준의 생활권)
- 종속 변수 : 주택 중위가격 (median_house_value)
- 설명 변수 : 지역의 인구·주택 정보 (인구수, 가구수, 주택 연식, 방 개수 등)

[주요 실습 내용]
- MLP (Multi-Layer Perceptron)
- 머신러닝 결과와 성능 비교

# Random Seed 고정
- 결과 재현하기 위함

In [2]:
%pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 MB 61.4 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 25.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 51.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.8/566.8 kB 34.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 63.1 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22/22 [tensorflow]2 [tensorflow]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import tensorflow as tf

# Random Seed 고정 : Python, NumPy, TensorFlow Seed를 한 번에 설정
SEED = 42
tf.keras.utils.set_random_seed(SEED)

=> 딥러닝에서 난수가 사용되는 부분

| 구분       | 설명                                        |
| ----------------- | ----------------------------------------- |
| **가중치 초기화**       | 학습 시작 시 뉴런의 Weight가 무작위로 초기화              |
| **데이터 Shuffle**   | Epoch마다 학습 데이터 순서를 섞음                     |
| **Mini-Batch 구성** | Shuffle에 따라 Batch에 들어가는 데이터가 달라짐          |
| **Dropout**       | 학습할 때 임의의 뉴런을 비활성화                       |

# 데이타 준비
- 분석 데이타 : 캘리포니아 집값 데이터셋
- 데이타 설명 : 캘리포니아의 작은 지역(Census Block Group)별 인구·주택 정보를 이용하여 주택 가격를 예측하기 위함
- 변수 설명

| 컬럼명                | 한글명      | 의미                      | 비고               |
| ------------------ | -------- | ----------------------- | ---------------- |
| longitude          | 경도       | 해당 지역의 경도               | 연속형              |
| latitude           | 위도       | 해당 지역의 위도               | 연속형              |
| housing_median_age | 주택 중위 연식 | 해당 지역 주택의 중위 연식(년)      | 연속형              |
| total_rooms        | 전체 방 개수  | 해당 지역의 전체 방(Room) 수     | 연속형              |
| total_bedrooms     | 전체 침실 개수 | 해당 지역의 전체 침실(Bedroom) 수 | 연속형              |
| population         | 인구수      | 해당 지역의 총 인구수            | 연속형              |
| households         | 가구수      | 해당 지역의 총 가구 수           | 연속형              |
| median_income      | 중위소득     | 해당 지역 가구의 중위소득          | 연속형              |
| ocean_proximity    | 바다와의 거리  | 해당 지역의 바다와의 위치 관계       | **범주형**          |
| median_house_value | 주택 중위가격  | 해당 지역의 주택 중위가격(달러)      | **종속변수(Target)** |

- ocean_proximity 값 의미

| 값            | 의미           |
| ------------ | ------------ |
| `<1H OCEAN`  | 바다까지 1시간 이내  |
| `INLAND`     | 내륙 지역        |
| `NEAR OCEAN` | 바다 인접 지역     |
| `NEAR BAY`   | 만(Bay) 인접 지역 |
| `ISLAND`     | 섬 지역         |






데이타 불러오기

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings

warnings.filterwarnings("ignore")  # 경고 메시지 무시
url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(url)
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


컬럼정보 확인

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.7 MB


[참고] info()에서 확인할 것:
  - 총 행/열 수
  - 각 컬럼의 데이터 타입
    - int : 연속형 (정수형)
    - float : 연속형 (실수형)
    - object : 보통 문자열 (범주형)
    - category : 범주형,  *범주 갯수가 적을 경우 메모리 효율화를 위해 object 타입 대신 사용, 그러나 object 을 사용해도 무방
    - datetime : 날짜형

# EDA

연속형 변수 기본 분포값 확인
 - 비즈니스관점에서 데이터 이상 여부 확인, 특히 음수(-), '0' 데이터 주의
 - Missing 건수 확인
 - 데이터 최소, 최대, 중심값 확인
 - 데이터 밀집도 확인

In [6]:
import numpy as np

# 연속형 변수 기본 분포 확인
df.select_dtypes(include=[np.number]).describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


범주형 변수 기본 분포값 확인
 - 비즈니스관점에서 데이터 이상 여부 확인
 - Missing 건수 확인
 - 데이터 밀집도 확인
 - 범주 갯수 확인

In [7]:
# 범주형 변수 자동 추출
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# 범주 건수 & 비율 계산
for col in cat_cols:
    print(f"\n[{col}]")
    dist = df[col].value_counts(dropna=False)
    ratio = df[col].value_counts(normalize=True, dropna=False) * 100
    result = pd.DataFrame({
        "count": dist,
        "ratio_%": ratio.round(2)
    })
    print(result)


[ocean_proximity]
                 count  ratio_%
ocean_proximity                
<1H OCEAN         9136    44.26
INLAND            6551    31.74
NEAR OCEAN        2658    12.88
NEAR BAY          2290    11.09
ISLAND               5     0.02


# 데이타 분할

X, y 분리

In [8]:
# X, y 분리
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

Data Partition
- 목적 : 모델이 학습에 사용하지 않은 새로운 데이터에서도 잘 동작하는지 검증하기 위함
 - Train  : Test 비율 (7 : 3)
- Train Set : 모델 학습용, Test Set : 모델 평가용
- 데이타 변환시 변환 통계량은 Train 데이터에서 계산하고, Test 데이터에는 그 기준을 적용해야 함 (Data Leakage 문제)

In [9]:
from sklearn.model_selection import train_test_split

# Train/Test 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Feature Transformation

Missing 처리
- 연속형: 중앙값 대체

In [10]:
import numpy as np
import pandas as pd

# 연속형 변수 선택
num_cols = X_train.select_dtypes(include=[np.number]).columns

# Train 기준 중앙값 계산
median_values = X_train[num_cols].median()

# Train, Test 모두 Train 기준 중앙값으로 대체
X_train[num_cols] = X_train[num_cols].fillna(median_values)
X_test[num_cols] = X_test[num_cols].fillna(median_values)

- 범주형: New Class 대체

In [11]:
# 범주형 변수 선택
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns

# 신규 범주(Unknown)로 대체
for col in cat_cols:

    # Train
    if X_train[col].dtype.name == "category":
        X_train[col] = X_train[col].cat.add_categories("Unknown")
    X_train[col] = X_train[col].fillna("Unknown")

    # Test
    if X_test[col].dtype.name == "category":
        X_test[col] = X_test[col].cat.add_categories("Unknown")
    X_test[col] = X_test[col].fillna("Unknown")

Scaling
   - Min-Max Scaling

In [12]:
from sklearn.preprocessing import MinMaxScaler

# 연속형 변수 추출
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Min-Max Scaling
scaler = MinMaxScaler()

# Train
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])

 # Test
X_test[num_cols] = scaler.transform(X_test[num_cols])

범주형 변수 -> 수치형 변수로 변환
- One-hot Encoding

In [13]:
# 범주형 변수 선택
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns

# Train Set
X_train = pd.get_dummies(
    X_train,
    columns=cat_cols,
    drop_first=True      # False : One-Hot Encoding, True : Dummy Encoding(k-1)
)

# Test Set
X_test = pd.get_dummies(
    X_test,
    columns=cat_cols,
    drop_first=True
)

# Test를 Train 컬럼 구조에 맞춤
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

#  [참고] 선형 회귀 모델
   - API 문서 : https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error

# 모델 정의
model = LinearRegression()

# 모델 학습
model.fit(X_train, y_train)

# 예측값 생성
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# 모델 평가
print(f"Train MAPE: {mean_absolute_percentage_error(y_train, train_pred) * 100:.3f}")
print(f"Test  MAPE: {mean_absolute_percentage_error(y_test, test_pred) * 100:.3f}")

Train MAPE: 28.555
Test  MAPE: 28.953


# MLP (Multi-Layer Perceptron)
 - 가장 기본적인 신경망 모형
 - API 문서 :  https://www.tensorflow.org/api_docs/python/tf/keras/Sequential

학습 및 예측

In [15]:
import time
from sklearn.metrics import mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

start_time = time.time()

# Random Seed 고정
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# 모델 정의, *Sequential: Layer를 쌓아 모델을 구성하는 방식
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),           # 은닉층 1번째, 64개 노드(뉴런), X_train.shape[1]: 피처갯수
    Dense(32, activation="relu"),                                            # 은닉층 2번째, 32개 노드(뉴런)
    Dense(1)                                                                 # 출력층, 예측값 1개
])

# 모델 컴파일
model.compile(
    optimizer=Adam(learning_rate=0.1),   # Optimizer: Adam, 학습률 : 0.1
    loss="mse"                           # 손실함수는 일반적으로 '0' 근처에서 안정적인 mse 를 사용
)

# EarlyStopping 정의
early_stop = EarlyStopping(
    monitor="val_loss",           # 학습 데이터가 아니라 검증 데이터 기준으로 멈춤
    patience=5,                   # 최저점 이후 5번 더 참았다가 멈춤
    restore_best_weights=True     # 가장 val_loss가 좋았던 시점의 가중치로 되돌림
)

# 모델 학습
model.fit(
    X_train,
    y_train,
    validation_split=0.2,     # validation set 비율
    epochs=50,                # 에포크수
    batch_size=32,            # 배치크기
    callbacks=[early_stop],   # EarlyStopping 조건 실행
    verbose=1                 # 학습 과정 출력
)

# 예측값 생성
train_pred = model.predict(X_train).ravel()
test_pred  = model.predict(X_test).ravel()

end_time = time.time()

# 모델 평가
print("\n=== MLP ===")
print(f"Train MAPE : {mean_absolute_percentage_error(y_train, train_pred) * 100:.3f}%")
print(f"Test  MAPE : {mean_absolute_percentage_error(y_test, test_pred) * 100:.3f}%")

# 소요 시간
print(f"Execution Time : {end_time - start_time:.2f} sec")

Epoch 1/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 406us/step - loss: 12225476608.0000 - val_loss: 5499879424.0000
Epoch 2/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 340us/step - loss: 5221436928.0000 - val_loss: 5237280768.0000
Epoch 3/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 336us/step - loss: 5043091456.0000 - val_loss: 5106524160.0000
Epoch 4/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 370us/step - loss: 4919696384.0000 - val_loss: 5001789440.0000
Epoch 5/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 311us/step - loss: 4818664960.0000 - val_loss: 4917235712.0000
Epoch 6/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 383us/step - loss: 4735932416.0000 - val_loss: 4850230784.0000
Epoch 7/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 375us/step - loss: 4669793280.0000 - val_loss: 4788599296.0000
Epoch 8/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 312us/step - loss: 4610033152.0000 - val_loss: 4738545664.0000
Epoch 9/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 306us/step - loss: 4556698112.0000 - val_loss: 4674124800.0000
Epoch 10/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 2

=> [참고] MLP 신경망 총 파라미터 갯수
- feature=12 (X_train.shape[1])
- 첫번째 층 : 12 * 64 + 64 = 832 (12: 피처갯수, 앞 64: 노드수, 뒤 64 : 편향(Bias) 갯수
- 두번째 층 : 64 * 32 + 32 = 2,080 (64: 첫번째 층 노드수, 32: 두번째 층 노드수)
- 출력 층 : 32 * 1 + 1 = 33 (32: 두번째 층 노드수, 1: 출력 노드수)
- 총 파라미터 갯수 : 832 + 2080 + 33 = 2,945
- (참고) 선형 회귀모형 파라미터 갯수 : 12 + 1 = 13

### [Self-Practice]
Q1) 위 신경망에 은닉층을 추가해서 깊은 신경망을 생성 후 모델 성능(Test MAPE)을 비교해 보세요. Ex) 은닉층 4개(128 → 64 → 32 → 16)   
Q2~5) 위 신경망에 아래 Hyperparameter 값 변화에 따른 각각의 모델 성능(Test MAPE) 과 소요 시간(Execution Time) 을 측정하고 비교해 보세요.  
> Q2) patience=5, 10, 20   
> Q3) learning_rate=0.1, 0.01, 0.001   
> Q4) epochs=50, 100, 200   
> Q5) batch_size=32, 16, 8

Q6) 위 성능 결과와 ML 모델 성능 결과 (XGBoost, LightGBM)와 비교해 보세요.

- [참고] 스케줄러 (Scheduler) 추가

In [ ]:
import time
from sklearn.metrics import mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

start_time = time.time()

# 모델 정의
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),           # 은닉층 1번째, 64개 노드(뉴런), X_train.shape[1]: 피처갯수
    Dense(32, activation="relu"),                                            # 은닉층 2번째, 32개 노드(뉴런)
    Dense(1)                                                                 # 출력층, 예측값 1개
])

# 모델 컴파일
model.compile(
    optimizer=Adam(learning_rate=0.1),   # Optimizer: Adam, 학습률 : 0.1
    loss="mse"                           # 손실함수는 일반적으로 '0' 근처에서 안정적인 mse 를 사용
)

# EarlyStopping 정의
early_stop = EarlyStopping(
    monitor="val_loss",           # 학습 데이터가 아니라 검증 데이터 기준으로 멈춤
    patience=5,                   # 최저점 이후 5번 더 참았다가 멈춤
    restore_best_weights=True     # 가장 val_loss가 좋았던 시점의 가중치로 되돌림
)

# 학습률 스케줄러 정의
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",   # 검증 손실을 모니터링
    factor=0.5,           # 학습률(learning_rate)을 50%로 감소
    patience=2,           # 2 Epoch 동안 개선이 없으면 학습률 감소
    min_lr=1e-6           # 학습률의 최솟값
)

# 모델 학습
model.fit(
    X_train,
    y_train,
    validation_split=0.2,     # validation set 비율
    epochs=50,                # 에포크수
    batch_size=32,            # 배치크기
    callbacks=[early_stop, lr_scheduler],  # EarlyStopping, 학습률 스케줄러 실행,
    verbose=1                 # 학습 과정 출력
)

# 예측값 생성
train_pred = model.predict(X_train).ravel()
test_pred  = model.predict(X_test).ravel()

end_time = time.time()

# 모델 평가
print("\n=== MLP ===")
print(f"Train MAPE : {mean_absolute_percentage_error(y_train, train_pred) * 100:.3f}%")
print(f"Test  MAPE : {mean_absolute_percentage_error(y_test, test_pred) * 100:.3f}%")

# 소요 시간
print(f"Execution Time : {end_time - start_time:.2f} sec")

# 시나리오 1

In [18]:
import time
from sklearn.metrics import mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

start_time = time.time()

# Random Seed 고정
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# 모델 정의, *Sequential: Layer를 쌓아 모델을 구성하는 방식
model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train.shape[1],)),           # 은닉층 1번째, 64개 노드(뉴런), X_train.shape[1]: 피처갯수
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),                                            # 은닉층 2번째, 32개 노드(뉴런)
    Dense(16, activation="relu"),
    Dense(1)                                                                 # 출력층, 예측값 1개
])

# 모델 컴파일
model.compile(
    optimizer=Adam(learning_rate=0.1),   # Optimizer: Adam, 학습률 : 0.1
    loss="mse"                           # 손실함수는 일반적으로 '0' 근처에서 안정적인 mse 를 사용
)

# EarlyStopping 정의
early_stop = EarlyStopping(
    monitor="val_loss",           # 학습 데이터가 아니라 검증 데이터 기준으로 멈춤
    patience=5,                   # 최저점 이후 5번 더 참았다가 멈춤
    restore_best_weights=True     # 가장 val_loss가 좋았던 시점의 가중치로 되돌림
)

# 모델 학습
model.fit(
    X_train,
    y_train,
    validation_split=0.2,     # validation set 비율
    epochs=50,                # 에포크수
    batch_size=32,            # 배치크기
    callbacks=[early_stop],   # EarlyStopping 조건 실행
    verbose=1                 # 학습 과정 출력
)

# 예측값 생성
train_pred = model.predict(X_train).ravel()
test_pred  = model.predict(X_test).ravel()

end_time = time.time()

# 모델 평가
print("\n=== MLP ===")
print(f"Train MAPE : {mean_absolute_percentage_error(y_train, train_pred) * 100:.3f}%")
print(f"Test  MAPE : {mean_absolute_percentage_error(y_test, test_pred) * 100:.3f}%")

# 소요 시간
print(f"Execution Time : {end_time - start_time:.2f} sec")

Epoch 1/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 455us/step - loss: 7479121920.0000 - val_loss: 5890694656.0000
Epoch 2/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 337us/step - loss: 5432783360.0000 - val_loss: 5753393664.0000
Epoch 3/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 334us/step - loss: 5222211584.0000 - val_loss: 5489822208.0000
Epoch 4/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 338us/step - loss: 5067240960.0000 - val_loss: 5235969536.0000
Epoch 5/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 337us/step - loss: 4952283136.0000 - val_loss: 5052793856.0000
Epoch 6/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 336us/step - loss: 4866734080.0000 - val_loss: 4841120768.0000
Epoch 7/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 336us/step - loss: 4800739840.0000 - val_loss: 4668366336.0000
Epoch 8/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 334us/step - loss: 4740548608.0000 - val_loss: 4566628864.0000
Epoch 9/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 334us/step - loss: 4685273600.0000 - val_loss: 4537303552.0000
Epoch 10/50
362/362 ━━━━━━━━━━━━━━━━━━━━ 0s 33

# 시나리오 2,3,4,5

#### 기존 값

=== MLP ===

Train MAPE : 25.526%

Test  MAPE : 25.934%


Q2. patience : 변화없음

Q3. learning_rate: 값 증가

Q4. 에포크: 값 하락

Q5. 배치사이즈: 값 증가

In [23]:
import time
from sklearn.metrics import mean_absolute_percentage_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

start_time = time.time()

# Random Seed 고정
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# 모델 정의, *Sequential: Layer를 쌓아 모델을 구성하는 방식
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),           # 은닉층 1번째, 64개 노드(뉴런), X_train.shape[1]: 피처갯수
    Dense(32, activation="relu"),                                            # 은닉층 2번째, 32개 노드(뉴런)
    Dense(1)                                                                 # 출력층, 예측값 1개
])

# 모델 컴파일
model.compile(
    optimizer=Adam(learning_rate=0.1),   # Optimizer: Adam, 학습률 : 0.1
    loss="mse"                           # 손실함수는 일반적으로 '0' 근처에서 안정적인 mse 를 사용
)

# EarlyStopping 정의
early_stop = EarlyStopping(
    monitor="val_loss",           # 학습 데이터가 아니라 검증 데이터 기준으로 멈춤
    patience=10,                   # 최저점 이후 5번 더 참았다가 멈춤
    restore_best_weights=True     # 가장 val_loss가 좋았던 시점의 가중치로 되돌림
)

# 모델 학습
model.fit(
    X_train,
    y_train,
    validation_split=0.2,     # validation set 비율
    epochs=50,                # 에포크수
    batch_size=64,            # 배치크기
    callbacks=[early_stop],   # EarlyStopping 조건 실행
    verbose=1                 # 학습 과정 출력
)

# 예측값 생성
train_pred = model.predict(X_train).ravel()
test_pred  = model.predict(X_test).ravel()

end_time = time.time()

# 모델 평가
print("\n=== MLP ===")
print(f"Train MAPE : {mean_absolute_percentage_error(y_train, train_pred) * 100:.3f}%")
print(f"Test  MAPE : {mean_absolute_percentage_error(y_test, test_pred) * 100:.3f}%")

# 소요 시간
print(f"Execution Time : {end_time - start_time:.2f} sec")

Epoch 1/50
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 561us/step - loss: 18644334592.0000 - val_loss: 7382831104.0000
Epoch 2/50
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 349us/step - loss: 6016470016.0000 - val_loss: 5564173312.0000
Epoch 3/50
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 339us/step - loss: 5252370432.0000 - val_loss: 5501212160.0000
Epoch 4/50
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 395us/step - loss: 5118542848.0000 - val_loss: 5462752768.0000
Epoch 5/50
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 461us/step - loss: 5013888512.0000 - val_loss: 5419468288.0000
Epoch 6/50
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 398us/step - loss: 4923782144.0000 - val_loss: 5375885312.0000
Epoch 7/50
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 338us/step - loss: 4840070144.0000 - val_loss: 5318283264.0000
Epoch 8/50
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 447us/step - loss: 4757244928.0000 - val_loss: 5239279104.0000
Epoch 9/50
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 329us/step - loss: 4676932608.0000 - val_loss: 5159579648.0000
Epoch 10/50
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 3

# Q6

In [26]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error

# 모델 정의
model = LGBMRegressor(
    n_estimators=40,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    verbosity=-1      # 로그 출력 안 함
)

# 모델 학습
model.fit(X_train, y_train)

# 예측값 생성
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# 모델 평가
print(f"Train MAPE: {mean_absolute_percentage_error(y_train, train_pred) * 100:.3f}%")
print(f"Test  MAPE: {mean_absolute_percentage_error(y_test, test_pred) * 100:.3f}%")

Train MAPE: 19.770%
Test  MAPE: 20.885%
